# Data Preparation & Cleaning: Aroma Jaya Coffee Shop

**Tujuan:** membersihkan data transaksi POS 3 cabang Aroma Jaya (Sudirman, Senopati, Gading Serpong) agar siap dianalisis di Excel.

**Keputusan bisnis yang dibantu:** mengatur staf dan stok sesuai hari dan jam sibuk.

In [1]:
import pandas as pd

RAW_PATH = "dataset_pos_aromajaya.csv"  

D:\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
df = pd.read_csv(RAW_PATH)
n_awal = len(df)
print(f"Baris mentah: {n_awal:,}  |  Kolom: {df.shape[1]}")
df.head(8)

Baris mentah: 42,643  |  Kolom: 10


,transaction_id,transaction_date,transaction_time,store_location,product_id,product_name,product_category,quantity,unit_price,tipe_order
0,TX00001,2026-01-01,07:00:18,Gading Serpong (Residential),106,Roti Bakar Bandung,Snack,1,18000,Dine-in
1,TX00002,2026-01-01,07:01:15,Senopati (Hangout),102,Americano,Coffee,2,25000,Takeaway
2,TX00003,2026-01-01,07:04:37,sudirman (office),101,Es Kopi Susu Gula Aren,Coffee,1,22000,takeaway
3,TX00004,2026-01-01,07:07:02,Gading Serpong (Residential),101,Es Kopi Susu Gula Aren,Coffee,1,22000,Dine-in
4,TX00005,2026-01-01,07:07:38,Gading Serpong (Residential),101,Es Kopi Susu Gula Aren,Coffee,1,22000,Dine-in
5,TX00005,2026-01-01,07:07:38,Gading Serpong (Residential),102,Americano,Coffee,1,25000,Dine-in
6,TX00005,2026-01-01,07:07:38,Gading Serpong (Residential),107,Es Teh Manis,Non-Coffee,4,10000,Dine-in
7,TX00006,2026-01-01,07:18:54,Sudirman (Office),104,Matcha Latte,Non-Coffee,1,28000,Dine-in


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 42643 entries, 0 to 42642
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   transaction_id    42643 non-null  str  
 1   transaction_date  42643 non-null  str  
 2   transaction_time  42643 non-null  str  
 3   store_location    42636 non-null  str  
 4   product_id        42643 non-null  int64
 5   product_name      42643 non-null  str  
 6   product_category  42643 non-null  str  
 7   quantity          42643 non-null  int64
 8   unit_price        42643 non-null  int64
 9   tipe_order        42637 non-null  str  
dtypes: int64(3), str(7)
memory usage: 6.3 MB


## Cleaning

### buang duplikat penuh
Baris yang sama persis di semua kolom (transaksi, produk, jumlah, waktu) dianggap tercatat dua kali, jadi satu dibuang.

In [4]:
n = len(df)
df = df.drop_duplicates()
print(f"Dibuang: {n - len(df)} baris")

Dibuang: 35 baris


### buang quantity tidak wajar
- `-2`: tidak ada transaksi asal untuk dicocokkan, jadi tidak bisa dipastikan sebagai retur.
- `99`: jauh di atas nilai wajar (maksimal 5), kemungkinan salah input.

In [5]:
df = df[df["quantity"] > 0]
df = df[df["quantity"] < 99]

print("Quantity sekarang:", sorted(df["quantity"].unique()))

Quantity sekarang: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


### seragamkan nama cabang
Pencocokan memakai kata kunci (huruf kecil) supaya semua variasi penulisan jadi satu nama baku. Baris yang kosong diisi `Unknown` (bukan dibuang), karena jumlahnya kecil dan baris tersebut masih berguna untuk analisis hari dan jam.

In [6]:
peta_cabang = {
    "sudirman": "Sudirman (Office)",
    "senopati": "Senopati (Hangout)",
    "gading": "Gading Serpong (Residential)",
}

def seragamkan_cabang(teks):
    if pd.isna(teks):
        return "Unknown"
    t = str(teks).lower()
    for kata, nama in peta_cabang.items():
        if kata in t:
            return nama
    return "Unknown"

df["store_location"] = df["store_location"].apply(seragamkan_cabang)
df["store_location"].value_counts()

store_location
Sudirman (Office)               19851
Senopati (Hangout)              14772
Gading Serpong (Residential)     7953
Unknown                             7
Name: count, dtype: int64

### seragamkan tipe order
`DINE IN`, `Dine-in` dijadikan `Dine-in`; `takeaway`, `Takeaway` dijadikan `Takeaway`; kosong menjadi `Unknown`.

In [7]:
def seragamkan_tipe(teks):
    if pd.isna(teks):
        return "Unknown"
    t = str(teks).lower().replace(" ", "").replace("-", "")
    if t == "dinein":
        return "Dine-in"
    if t == "takeaway":
        return "Takeaway"
    return "Unknown"

df["tipe_order"] = df["tipe_order"].apply(seragamkan_tipe)
df["tipe_order"].value_counts()

tipe_order
Dine-in     22090
Takeaway    20487
Unknown         6
Name: count, dtype: int64

Cek: apakah penyeragaman teks memunculkan duplikat baru (baris yang tadinya beda penulisan, sekarang sama)?

In [8]:
n_dup_baru = int(df.duplicated().sum())
print("Duplikat baru setelah teks diseragamkan:", n_dup_baru)

Duplikat baru setelah teks diseragamkan: 0


### tambah kolom untuk analisis

1. `hour` = jam transaksi (7 sampai 21)
2. `day_num` = 1 = Senin sampai 7 = Minggu
3. `day_name` = nama hari
4. `month` = bulan (YYYY-MM)
5. `revenue` = `quantity` x `unit_price` 

In [9]:
df["transaction_date"] = pd.to_datetime(df["transaction_date"])
df["hour"] = df["transaction_time"].str[:2].astype(int)
df["day_num"] = df["transaction_date"].dt.dayofweek + 1
nama_hari = {1: "Senin", 2: "Selasa", 3: "Rabu", 4: "Kamis", 5: "Jumat", 6: "Sabtu", 7: "Minggu"}
df["day_name"] = df["day_num"].map(nama_hari)
df["month"] = df["transaction_date"].dt.strftime("%Y-%m")
df["revenue"] = df["quantity"] * df["unit_price"]

df = df.sort_values(["transaction_date", "transaction_time", "transaction_id"]).reset_index(drop=True)
df.head()

,transaction_id,transaction_date,transaction_time,store_location,product_id,product_name,product_category,quantity,unit_price,tipe_order,hour,day_num,day_name,month,revenue
0,TX00001,2026-01-01,07:00:18,Gading Serpong (Residential),106,Roti Bakar Bandung,Snack,1,18000,Dine-in,7,4,Kamis,2026-01,18000
1,TX00002,2026-01-01,07:01:15,Senopati (Hangout),102,Americano,Coffee,2,25000,Takeaway,7,4,Kamis,2026-01,50000
2,TX00003,2026-01-01,07:04:37,Sudirman (Office),101,Es Kopi Susu Gula Aren,Coffee,1,22000,Takeaway,7,4,Kamis,2026-01,22000
3,TX00004,2026-01-01,07:07:02,Gading Serpong (Residential),101,Es Kopi Susu Gula Aren,Coffee,1,22000,Dine-in,7,4,Kamis,2026-01,22000
4,TX00005,2026-01-01,07:07:38,Gading Serpong (Residential),101,Es Kopi Susu Gula Aren,Coffee,1,22000,Dine-in,7,4,Kamis,2026-01,22000


## Cek hasil dan keterbatasan

Satu struk seharusnya hanya satu cabang. Cek berapa transaksi yang tercatat di lebih dari satu cabang. Baris ini **dibiarkan apa adanya** (keputusan analis), tetapi harus dicatat sebagai keterbatasan.

In [10]:
cab_per_trx = df[df["store_location"] != "Unknown"].groupby("transaction_id")["store_location"].nunique()
n_trx_multi = int((cab_per_trx > 1).sum())
n_trx_total = df["transaction_id"].nunique()
print(f"Transaksi di lebih dari satu cabang: {n_trx_multi:,} dari {n_trx_total:,} ({n_trx_multi / n_trx_total:.1%})")

print("\nSisa baris kosong:", int(df.isna().sum().sum()))
print("Rentang tanggal:", df["transaction_date"].min().date(), "s.d.", df["transaction_date"].max().date())

Transaksi di lebih dari satu cabang: 831 dari 29,987 (2.8%)

Sisa baris kosong: 0
Rentang tanggal: 2026-01-01 s.d. 2026-06-29


## Simpan data bersih

In [11]:
df_simpan = df.copy()
df_simpan["transaction_date"] = df_simpan["transaction_date"].dt.strftime("%Y-%m-%d")
df_simpan.to_csv("data_pos_aromajaya_clean.csv", index=False)

## Ringkasan

Data mentah **42.643 baris** menjadi **42.583 baris** bersih (60 baris dibuang, 0,14%). Teks cabang dan tipe order sudah seragam, dan kolom `hour`, `day_num`, `day_name`, `month`, `revenue` siap dipakai